#DEMANDAS TI

### CONFIGURAÇÃO E CARREGAMENTO DE DATASET

In [118]:
# ============================================================
# Importa as bibliotecas do Python
# ============================================================

# Importar o pandas
import pandas as pd
import numpy as np

# Importar o LabelEncoder da biblioteca scikit-learn
from sklearn.preprocessing import LabelEncoder

# Importar a função de divisão de treino e teste
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score

# Modelos
from sklearn.ensemble import RandomForestClassifier
from xgboost          import XGBClassifier
from sklearn.metrics  import (accuracy_score, precision_score,
                               recall_score, f1_score,
                               classification_report,  confusion_matrix, roc_auc_score)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

import warnings
warnings.filterwarnings('ignore')




In [119]:
# ============================================================
# Carregar o dataset a partir do arquivo Excel
# ============================================================

# url do arquivo no GitHub
url_dataset = "https://raw.githubusercontent.com/gilbertoag2007/machine-learning-demandas-ti/main/DEMANDAS_DOWNSTREAM_V4.xlsx"

# Cria um dataframe com o conteúdo do dataset
colunas_desejadas = ["ID_DEMANDA", "SISTEMA", "NOME_EQUIPE", "TIPO_DEMANDA", "NUM_DEMANDA", "ANO_DEMANDA", "SITUACAO_DEMANDA","DAT_INICIAL_CLASSIFICACAO", "DAT_PREVISTA_INI_DEMANDA","DAT_PREVISTA_INI_REQUISITOS", "DAT_REAL_INI_REQ", "DAT_PREVISTA_FIM_REQUISITOS", "DAT_REAL_FIM_REQ", "DIAS_ATRASO_REQUISITOS", "DAT_PREVISTA_INI_DESENV", "DAT_REAL_INI_DESENV", "DAT_PREVISTA_FIM_DESENV", "DAT_REAL_FIM_DESENV","DIAS_ATRASO_DESENVOLVIMENTO", "DAT_PREVISTA_FIM_DEMANDA", "CLASSIFICACAO"]
df_original = pd.read_excel(url_dataset, usecols=colunas_desejadas )

# Lista as 5 primeiras colunas do dataframe.

df_original.head()

,ID_DEMANDA,SISTEMA,NOME_EQUIPE,TIPO_DEMANDA,NUM_DEMANDA,ANO_DEMANDA,SITUACAO_DEMANDA,DAT_INICIAL_CLASSIFICACAO,DAT_PREVISTA_INI_DEMANDA,DAT_PREVISTA_INI_REQUISITOS,...,DAT_PREVISTA_FIM_REQUISITOS,DAT_REAL_FIM_REQ,DIAS_ATRASO_REQUISITOS,DAT_PREVISTA_INI_DESENV,DAT_REAL_INI_DESENV,DAT_PREVISTA_FIM_DESENV,DAT_REAL_FIM_DESENV,DIAS_ATRASO_DESENVOLVIMENTO,DAT_PREVISTA_FIM_DEMANDA,CLASSIFICACAO
0,22391,CSA,Corporativo,ORIENTAÇÃO,797,2026,FINALIZADA,2026-05-14 16:00:50,2026-05-15,NaT,...,NaT,NaT,NaN,2026-05-15,2026-05-15 11:08:57,2026-05-18,2026-05-15 12:02:28,0,2026-05-18,NO PRAZO
1,22381,DPP,Downstream,BUG IMPEDITIVO,787,2026,FINALIZADA,2026-05-13 19:00:43,2026-05-14,NaT,...,NaT,NaT,NaN,2026-05-14,2026-05-15 16:34:46,2026-05-15,2026-05-15 18:24:44,0,2026-05-15,NO PRAZO
2,22366,I-SIMP (DPP),Downstream,BUG NÃO IMPEDITIVO,772,2026,FINALIZADA,2026-05-11 12:00:35,2026-05-12,NaT,...,NaT,NaT,NaN,2026-05-12,2026-05-12 14:17:40,2026-05-18,2026-05-13 10:13:32,0,2026-05-18,NO PRAZO
3,22360,DPP,Downstream,BUG NÃO IMPEDITIVO,766,2026,FINALIZADA,2026-05-08 14:00:27,2026-05-11,NaT,...,NaT,NaT,NaN,2026-05-11,2026-05-11 09:55:40,2026-05-15,2026-05-15 12:19:49,0,2026-05-15,NO PRAZO
4,22327,SIGAF,Downstream,MELHORIA PEQUENA,733,2026,FINALIZADA,2026-05-05 15:01:16,2026-05-08,NaT,...,NaT,NaT,NaN,2026-05-08,2026-05-11 15:49:19,2026-05-14,2026-05-11 16:27:17,0,2026-05-14,NO PRAZO


##AJUSTES INICIAIS NO DATAFRAME

In [120]:
# ============================================================
# TÉCNICA: Label Encoding (Codificação de Rótulos)
# ============================================================

# Cria uma cópia do dataframe original para preservá-lo intacto
# Todas as alterações serão feitas apenas no df_ajustado
df_ajustado = df_original.copy()

# Cria uma nova coluna numérica baseada na coluna STATUS_FINAL
# map() substitui cada valor categórico pelo número correspondente
df_ajustado['CLASSIFICACAO_FINAL_NUM'] = df_ajustado['CLASSIFICACAO'].map({
    'ATRASO'         : 1,
    'NO PRAZO': 0
})

# Variavel Target
target = "CLASSIFICACAO_FINAL_NUM"



In [121]:
# ============================================================
# TÉCNICA: One-Hot Encoding
# ============================================================

# Aplicar One-Hot Encoding na coluna SISTEMA
# pd.get_dummies() cria uma coluna binária (0 ou 1) para cada sistema único
# dtype=int garante que os valores sejam inteiros ao invés de booleanos
# Aplicar nas colunas categóricas sem ordem natural
for coluna in ['SISTEMA', 'TIPO_DEMANDA']:
    dummies = pd.get_dummies(df_ajustado[coluna], prefix=coluna, dtype=int)
    df_ajustado = pd.concat([df_ajustado, dummies], axis=1)
    df_ajustado = df_ajustado.drop(columns=[coluna])


In [122]:

# ============================================================
# CONVERTER COLUNAS DE DATA PARA DATETIME
# Necessário para realizar operações matemáticas entre datas
# ============================================================
colunas_data = [
    'DAT_INICIAL_CLASSIFICACAO',
    'DAT_PREVISTA_INI_DEMANDA',
    'DAT_PREVISTA_INI_REQUISITOS',
    'DAT_REAL_INI_REQ',
    'DAT_PREVISTA_FIM_REQUISITOS',
    'DAT_REAL_FIM_REQ',
    'DAT_PREVISTA_INI_DESENV',
    'DAT_REAL_INI_DESENV',
    'DAT_PREVISTA_FIM_DESENV',
    'DAT_REAL_FIM_DESENV',
    'DAT_PREVISTA_FIM_DEMANDA'
]

for coluna in colunas_data:
    df_ajustado[coluna] = pd.to_datetime(
        df_ajustado[coluna], dayfirst=True, errors='coerce'
    )


In [123]:
# FEATURE ENGENIER

# Inclusão de feature para quantidade de dias de atraso no inicio do desenvolvimento.
df_ajustado['ATRASO_INICIO_DESENV']  = (df_ajustado['DAT_REAL_INI_DESENV']      - df_ajustado['DAT_PREVISTA_INI_DESENV']).dt.days

# Atribui -1 quando DIAS_ATRASO_REQUISITOS for nulo
df_ajustado['DIAS_ATRASO_REQUISITOS'] = df_ajustado['DIAS_ATRASO_REQUISITOS'].fillna(-1)

df_ajustado.head(50)

,ID_DEMANDA,NOME_EQUIPE,NUM_DEMANDA,ANO_DEMANDA,SITUACAO_DEMANDA,DAT_INICIAL_CLASSIFICACAO,DAT_PREVISTA_INI_DEMANDA,DAT_PREVISTA_INI_REQUISITOS,DAT_REAL_INI_REQ,DAT_PREVISTA_FIM_REQUISITOS,...,SISTEMA_SIGAF,SISTEMA_SIMP,SISTEMA_SRD - GLP,SISTEMA_SRD - PR,TIPO_DEMANDA_BUG IMPEDITIVO,TIPO_DEMANDA_BUG NÃO IMPEDITIVO,TIPO_DEMANDA_MELHORIA MÉDIA,TIPO_DEMANDA_MELHORIA PEQUENA,TIPO_DEMANDA_ORIENTAÇÃO,ATRASO_INICIO_DESENV
0,22391,Corporativo,797,2026,FINALIZADA,2026-05-14 16:00:50,2026-05-15,NaT,NaT,NaT,...,0,0,0,0,0,0,0,0,1,0
1,22381,Downstream,787,2026,FINALIZADA,2026-05-13 19:00:43,2026-05-14,NaT,NaT,NaT,...,0,0,0,0,1,0,0,0,0,1
2,22366,Downstream,772,2026,FINALIZADA,2026-05-11 12:00:35,2026-05-12,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,0
3,22360,Downstream,766,2026,FINALIZADA,2026-05-08 14:00:27,2026-05-11,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,0
4,22327,Downstream,733,2026,FINALIZADA,2026-05-05 15:01:16,2026-05-08,NaT,NaT,NaT,...,1,0,0,0,0,0,0,1,0,3
5,22320,Downstream,726,2026,FINALIZADA,2026-05-04 14:11:25,2026-05-07,NaT,NaT,NaT,...,0,0,0,0,0,0,0,1,0,-3
6,22312,Downstream,718,2026,FINALIZADA,2026-04-30 12:01:00,2026-05-04,NaT,NaT,NaT,...,0,0,0,0,1,0,0,0,0,1
7,22307,Downstream,713,2026,FINALIZADA,2026-04-29 20:00:12,2026-04-30,NaT,NaT,NaT,...,0,0,0,0,0,0,0,0,1,0
8,22302,Corporativo,708,2026,FINALIZADA,2026-04-29 15:00:57,2026-04-30,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,4
9,22298,Corporativo,704,2026,FINALIZADA,2026-04-29 12:00:57,2026-04-30,NaT,NaT,NaT,...,0,0,0,0,0,1,0,0,0,0


In [124]:
# Lista colunas do dataset
print(df_ajustado.columns.tolist())

['ID_DEMANDA', 'NOME_EQUIPE', 'NUM_DEMANDA', 'ANO_DEMANDA', 'SITUACAO_DEMANDA', 'DAT_INICIAL_CLASSIFICACAO', 'DAT_PREVISTA_INI_DEMANDA', 'DAT_PREVISTA_INI_REQUISITOS', 'DAT_REAL_INI_REQ', 'DAT_PREVISTA_FIM_REQUISITOS', 'DAT_REAL_FIM_REQ', 'DIAS_ATRASO_REQUISITOS', 'DAT_PREVISTA_INI_DESENV', 'DAT_REAL_INI_DESENV', 'DAT_PREVISTA_FIM_DESENV', 'DAT_REAL_FIM_DESENV', 'DIAS_ATRASO_DESENVOLVIMENTO', 'DAT_PREVISTA_FIM_DEMANDA', 'CLASSIFICACAO', 'CLASSIFICACAO_FINAL_NUM', 'SISTEMA_CSA', 'SISTEMA_DFe - Documento de Fiscalização Eletrônico', 'SISTEMA_DPP ', 'SISTEMA_I-SIMP (DPP)', 'SISTEMA_RENOVACALC', 'SISTEMA_SIGAF', 'SISTEMA_SIMP', 'SISTEMA_SRD - GLP', 'SISTEMA_SRD - PR', 'TIPO_DEMANDA_BUG IMPEDITIVO', 'TIPO_DEMANDA_BUG NÃO IMPEDITIVO', 'TIPO_DEMANDA_MELHORIA MÉDIA', 'TIPO_DEMANDA_MELHORIA PEQUENA', 'TIPO_DEMANDA_ORIENTAÇÃO', 'ATRASO_INICIO_DESENV']


In [125]:
# Exclui as colunas não necessárias para predição.

# Colunas One-Hot Encoding geradas para SISTEMA e TIPO_DEMANDA
colunas_sistema = [col for col in df_ajustado.columns if col.startswith('SISTEMA_')]
colunas_tipo    = [col for col in df_ajustado.columns if col.startswith('TIPO_DEMANDA_')]

# Features numéricas
colunas_numericas = [
    'ATRASO_INICIO_DESENV',
    'DIAS_ATRASO_REQUISITOS'
]

# Concatena todas as colunas necessárias
colunas_modelo = colunas_sistema + colunas_tipo + colunas_numericas + [target]

# Filtra o dataset
df_ajustado = df_ajustado[colunas_modelo]

# Separa features e target
X = df_ajustado.drop(columns=[target])
y = df_ajustado[target]

print(f'Features: {X.shape[1]} colunas')
print(f'Registros: {X.shape[0]}')
print(f'\nColunas utilizadas:\n{X.columns.tolist()}')
print(f'\nDistribuição do target:\n{y.value_counts()}')

Features: 16 colunas
Registros: 1632

Colunas utilizadas:
['SISTEMA_CSA', 'SISTEMA_DFe - Documento de Fiscalização Eletrônico', 'SISTEMA_DPP ', 'SISTEMA_I-SIMP (DPP)', 'SISTEMA_RENOVACALC', 'SISTEMA_SIGAF', 'SISTEMA_SIMP', 'SISTEMA_SRD - GLP', 'SISTEMA_SRD - PR', 'TIPO_DEMANDA_BUG IMPEDITIVO', 'TIPO_DEMANDA_BUG NÃO IMPEDITIVO', 'TIPO_DEMANDA_MELHORIA MÉDIA', 'TIPO_DEMANDA_MELHORIA PEQUENA', 'TIPO_DEMANDA_ORIENTAÇÃO', 'ATRASO_INICIO_DESENV', 'DIAS_ATRASO_REQUISITOS']

Distribuição do target:
CLASSIFICACAO_FINAL_NUM
0    1470
1     162
Name: count, dtype: int64


In [126]:
# Exibir resumo das colunas geradas e seus tipos
print('📊 Colunas do dataframe ajustado:')
print(df_ajustado.dtypes)
print(f'\n✅ Shape final: {df_ajustado.shape[0]} linhas x {df_ajustado.shape[1]} colunas')


📊 Colunas do dataframe ajustado:
SISTEMA_CSA                                             int64
SISTEMA_DFe - Documento de Fiscalização Eletrônico      int64
SISTEMA_DPP                                             int64
SISTEMA_I-SIMP (DPP)                                    int64
SISTEMA_RENOVACALC                                      int64
SISTEMA_SIGAF                                           int64
SISTEMA_SIMP                                            int64
SISTEMA_SRD - GLP                                       int64
SISTEMA_SRD - PR                                        int64
TIPO_DEMANDA_BUG IMPEDITIVO                             int64
TIPO_DEMANDA_BUG NÃO IMPEDITIVO                         int64
TIPO_DEMANDA_MELHORIA MÉDIA                             int64
TIPO_DEMANDA_MELHORIA PEQUENA                           int64
TIPO_DEMANDA_ORIENTAÇÃO                                 int64
ATRASO_INICIO_DESENV                                    int64
DIAS_ATRASO_REQUISITOS               

#ANALISE DOS DADOS

In [127]:
# ============================================================
# VERIFICAR O BALANCEAMENTO DA COLUNA TARGET
# ============================================================

balanceamento = df_ajustado[target].value_counts()
percentual    = df_ajustado[target].value_counts(normalize=True) * 100

# Exibir resultado
print('Distribuição da variável target:\n')
print(f'🟢 DENTRO DO PRAZO (0): {balanceamento[0]} registros ({percentual[0]:.1f}%)')
print(f'🔴 ATRASO          (1): {balanceamento[1]} registros ({percentual[1]:.1f}%)')

Distribuição da variável target:

🟢 DENTRO DO PRAZO (0): 1470 registros (90.1%)
🔴 ATRASO          (1): 162 registros (9.9%)


#TESTANDO OS MODELOS

In [128]:


# ----------------------------------------------------------
# Dividir em treino (80%) e teste (20%)
# stratify=y garante que a proporção de 0 e 1 seja mantida
# igual nos dois conjuntos — essencial para dados desbalanceados
# random_state=42 garante que a divisão seja reproduzível
# ----------------------------------------------------------
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size    = 0.20,
    stratify     = y,
    random_state = 42
)

# Exibir o resultado da divisão
print(f'Total de registros  : {len(X)}')
print(f'Registros de treino : {len(X_treino)} ({len(X_treino)/len(X)*100:.1f}%)')
print(f'Registros de teste  : {len(X_teste)} ({len(X_teste)/len(X)*100:.1f}%)')
print(f'\nDistribuição do target no treino:\n{y_treino.value_counts()}')
print(f'\nDistribuição do target no teste:\n{y_teste.value_counts()}')

Total de registros  : 1632
Registros de treino : 1305 (80.0%)
Registros de teste  : 327 (20.0%)

Distribuição do target no treino:
CLASSIFICACAO_FINAL_NUM
0    1175
1     130
Name: count, dtype: int64

Distribuição do target no teste:
CLASSIFICACAO_FINAL_NUM
0    295
1     32
Name: count, dtype: int64


In [129]:
# ─────────────────────────────────────────────
# 2. SMOTE — cria amostras sintéticas da classe minoritária
#    Só aplicar no conjunto de TREINO, nunca no teste!
# ─────────────────────────────────────────────
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_treino, y_treino)

print(f"Antes do SMOTE : {y_treino.value_counts().to_dict()}")
print(f"Depois do SMOTE: {y_train_bal.value_counts().to_dict()}")

Antes do SMOTE : {0: 1175, 1: 130}
Depois do SMOTE: {0: 1175, 1: 1175}


In [130]:
# ── 2. Divisão treino e teste ─────────────────────────────────────────────────
# stratify=y garante proporção 80/20 em ambos os conjuntos
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f'Treino : {X_train.shape[0]} registros')
print(f'Teste  : {X_test.shape[0]} registros')
print(f'\nDistribuição no treino:\n{y_train.value_counts()}')
print(f'\nDistribuição no teste:\n{y_test.value_counts()}')

Treino : 1305 registros
Teste  : 327 registros

Distribuição no treino:
CLASSIFICACAO_FINAL_NUM
0    1175
1     130
Name: count, dtype: int64

Distribuição no teste:
CLASSIFICACAO_FINAL_NUM
0    295
1     32
Name: count, dtype: int64


In [131]:
# ── 3. Definição dos modelos e variações de balanceamento ─────────────────────
modelos = {

    # Baseline simples — sem balanceamento
    'Regressão Logística (sem balanceamento)': (
        LogisticRegression(max_iter=1000, random_state=42),
        'sem_smote', None
    ),

    # Regressão Logística com penalização da classe minoritária
    'Regressão Logística (class_weight)': (
        LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'sem_smote', None
    ),

    # Random Forest sem balanceamento
    'Random Forest (sem balanceamento)': (
        RandomForestClassifier(n_estimators=200, random_state=42),
        'sem_smote', None
    ),

    # Random Forest com penalização
    'Random Forest (class_weight)': (
        RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42),
        'sem_smote', None
    ),

    # Random Forest com SMOTE
    'Random Forest (SMOTE)': (
        RandomForestClassifier(n_estimators=200, random_state=42),
        'smote', None
    ),

    # XGBoost sem balanceamento — melhores parâmetros encontrados pelo GridSearch
    'XGBoost (sem balanceamento)': (
        XGBClassifier(
            colsample_bytree=1.0,
            learning_rate=0.05,
            max_depth=6,
            n_estimators=300,
            scale_pos_weight=1,
            subsample=1.0,
            eval_metric='logloss',
            random_state=42
        ),
        'sem_smote', None
    ),

    # XGBoost com scale_pos_weight — melhores parâmetros encontrados pelo GridSearch
    'XGBoost (scale_pos_weight)': (
        XGBClassifier(
            colsample_bytree=0.6,
            learning_rate=0.01,
            max_depth=4,
            n_estimators=200,
            scale_pos_weight=9,
            subsample=1.0,
            eval_metric='logloss',
            random_state=42
        ),
        'sem_smote', None
    ),

    # XGBoost com SMOTE — melhores parâmetros encontrados pelo GridSearch
    'XGBoost (SMOTE)': (
        XGBClassifier(
            colsample_bytree=0.6,
            learning_rate=0.1,
            max_depth=5,
            n_estimators=200,
            subsample=1.0,
            eval_metric='logloss',
            random_state=42
        ),
        'smote', None
    ),
}

In [132]:
# ── 4. Treinamento e avaliação ─────────────────────────────────────────────────
THRESHOLD = 0.20  # threshold definido após análise de negócio

resultados = []

for nome, (modelo, estrategia, _) in modelos.items():

    # Aplica SMOTE apenas no treino quando indicado
    if estrategia == 'smote':
        X_treino_bal, y_treino_bal = SMOTE(random_state=42).fit_resample(X_train, y_train)
    else:
        X_treino_bal, y_treino_bal = X_train, y_train

    # Treina o modelo
    modelo.fit(X_treino_bal, y_treino_bal)

    # Gera probabilidades no conjunto de teste
    y_pred_prob = modelo.predict_proba(X_test)[:, 1]

    # Aplica o threshold definido no lugar do padrão 0.50
    y_pred = (y_pred_prob >= THRESHOLD).astype(int)

    # Métricas focadas na classe minoritária ATRASO
    report = classification_report(y_test, y_pred, output_dict=True)
    auc    = roc_auc_score(y_test, y_pred_prob)

    # Identifica qual label corresponde a ATRASO (1 ou 0 dependendo do LabelEncoder)
    classe_atraso = str(y_test.unique().max())

    resultados.append({
        'Modelo'           : nome,
        'Threshold'        : THRESHOLD,
        'F1 ATRASO'        : round(report[classe_atraso]['f1-score'], 3),
        'Precision ATRASO' : round(report[classe_atraso]['precision'], 3),
        'Recall ATRASO'    : round(report[classe_atraso]['recall'], 3),
        'AUC-ROC'          : round(auc, 3),
        'Acurácia'         : round(report['accuracy'], 3),
    })

    print(f'\n{"="*60}')
    print(f'Modelo: {nome}  |  Threshold: {THRESHOLD}')
    print(f'{"="*60}')
    print(classification_report(y_test, y_pred, target_names=['NO PRAZO', 'ATRASO']))
    print(f'AUC-ROC: {auc:.3f}')
    print(f'Matriz de Confusão:\n{confusion_matrix(y_test, y_pred)}')


Modelo: Regressão Logística (sem balanceamento)  |  Threshold: 0.2
              precision    recall  f1-score   support

    NO PRAZO       0.95      0.96      0.96       295
      ATRASO       0.60      0.56      0.58        32

    accuracy                           0.92       327
   macro avg       0.78      0.76      0.77       327
weighted avg       0.92      0.92      0.92       327

AUC-ROC: 0.879
Matriz de Confusão:
[[283  12]
 [ 14  18]]

Modelo: Regressão Logística (class_weight)  |  Threshold: 0.2
              precision    recall  f1-score   support

    NO PRAZO       0.98      0.48      0.65       295
      ATRASO       0.16      0.91      0.27        32

    accuracy                           0.52       327
   macro avg       0.57      0.69      0.46       327
weighted avg       0.90      0.52      0.61       327

AUC-ROC: 0.882
Matriz de Confusão:
[[142 153]
 [  3  29]]

Modelo: Random Forest (sem balanceamento)  |  Threshold: 0.2
              precision    recall  f1

**Precision** — Dos alertas de ATRASO que o modelo disparou, quantos eram atrasos de verdade.

"Quando o modelo grita atraso, eu posso confiar?"

**Recall** — De todos os atrasos reais que aconteceram, quantos o modelo conseguiu detectar.

"O modelo está deixando atrasos passarem sem alertar?"

**F1** — Média entre Precision e Recall. Útil para comparar modelos quando os dois importam.

"Equilíbrio geral do modelo."

In [133]:
# ── 5. Tabela comparativa dos resultados ──────────────────────────────────────
# Ordenada pelo F1-Score da classe ATRASO — métrica mais importante
df_resultados = pd.DataFrame(resultados).sort_values('F1 ATRASO', ascending=False)

print('\n\n══════════════════════════════════════════════════════════════')
print('COMPARATIVO GERAL — ordenado por F1 ATRASO')
print('══════════════════════════════════════════════════════════════')
print(df_resultados.to_string(index=False))



══════════════════════════════════════════════════════════════
COMPARATIVO GERAL — ordenado por F1 ATRASO
══════════════════════════════════════════════════════════════
                                 Modelo  Threshold  F1 ATRASO  Precision ATRASO  Recall ATRASO  AUC-ROC  Acurácia
            XGBoost (sem balanceamento)        0.2      0.655             0.783          0.562    0.836     0.942
      Random Forest (sem balanceamento)        0.2      0.610             0.667          0.562    0.821     0.930
Regressão Logística (sem balanceamento)        0.2      0.581             0.600          0.562    0.879     0.920
                        XGBoost (SMOTE)        0.2      0.442             0.333          0.656    0.811     0.838
                  Random Forest (SMOTE)        0.2      0.422             0.328          0.594    0.810     0.841
           Random Forest (class_weight)        0.2      0.365             0.253          0.656    0.804     0.777
     Regressão Logística (class

In [134]:
# Teste com valores reais


THRESHOLD = 0.20

# ── Registros de demandas em andamento para teste ─────────────────────────────
# Preencha com os dados reais das demandas em andamento
# ATRASO_INICIO_DESENV: positivo = atrasou, negativo = adiantado, 0 = no prazo
# DIAS_ATRASO_REQUISITOS: -1 = sem etapa de requisitos, 0 = no prazo, >0 = atrasou

novos_registros = pd.DataFrame([
    {
        'SISTEMA'                : 'SIGAF',
        'TIPO_DEMANDA'           : 'MELHORIA MÉDIA',
        'ATRASO_INICIO_DESENV'   : 5,
        'DIAS_ATRASO_REQUISITOS' : 3,
    },
    {
        'SISTEMA'                : 'SIMP',
        'TIPO_DEMANDA'           : 'BUG IMPEDITIVO',
        'ATRASO_INICIO_DESENV'   : 0,
        'DIAS_ATRASO_REQUISITOS' : -1,
    },
    {
        'SISTEMA'                : 'CSA',
        'TIPO_DEMANDA'           : 'MELHORIA PEQUENA',
        'ATRASO_INICIO_DESENV'   : 2,
        'DIAS_ATRASO_REQUISITOS' : -1,
    },
    {
        'SISTEMA'                : 'RENOVACALC',
        'TIPO_DEMANDA'           : 'MELHORIA MÉDIA',
        'ATRASO_INICIO_DESENV'   : 8,
        'DIAS_ATRASO_REQUISITOS' : 6,
    },
    {
        'SISTEMA'                : 'SRD - GLP',
        'TIPO_DEMANDA'           : 'BUG NÃO IMPEDITIVO',
        'ATRASO_INICIO_DESENV'   : 1,
        'DIAS_ATRASO_REQUISITOS' : -1,
    },
])

# ── Aplicar o mesmo One-Hot Encoding do treino ────────────────────────────────
novos_dummies   = pd.get_dummies(novos_registros, columns=['SISTEMA', 'TIPO_DEMANDA'])

# Alinha as colunas com as do modelo — preenche com 0 colunas ausentes
novos_alinhados = novos_dummies.reindex(columns=X.columns, fill_value=0)

# ── Gerar probabilidades e classificação ──────────────────────────────────────
probabilidades  = modelo_rf.predict_proba(novos_alinhados)[:, 1]
classificacoes  = ['ATRASO' if p >= THRESHOLD else 'NO PRAZO' for p in probabilidades]

# ── Resultado final ───────────────────────────────────────────────────────────
resultado = novos_registros.copy()
resultado['PROB_ATRASO']    = [f'{p:.1%}' for p in probabilidades]
resultado['CLASSIFICACAO']  = classificacoes
resultado['RISCO']          = ['🔴 ALTO' if p >= 0.5
                                else '🟡 MÉDIO' if p >= THRESHOLD
                                else '🟢 BAIXO'
                                for p in probabilidades]

print(resultado[['SISTEMA','TIPO_DEMANDA','ATRASO_INICIO_DESENV',
                 'DIAS_ATRASO_REQUISITOS','PROB_ATRASO',
                 'CLASSIFICACAO','RISCO']].to_string(index=False))



   SISTEMA       TIPO_DEMANDA  ATRASO_INICIO_DESENV  DIAS_ATRASO_REQUISITOS PROB_ATRASO CLASSIFICACAO   RISCO
     SIGAF     MELHORIA MÉDIA                     5                       3       46.8%        ATRASO 🟡 MÉDIO
      SIMP     BUG IMPEDITIVO                     0                      -1        7.6%      NO PRAZO 🟢 BAIXO
       CSA   MELHORIA PEQUENA                     2                      -1        1.1%      NO PRAZO 🟢 BAIXO
RENOVACALC     MELHORIA MÉDIA                     8                       6       47.0%        ATRASO 🟡 MÉDIO
 SRD - GLP BUG NÃO IMPEDITIVO                     1                      -1        0.0%      NO PRAZO 🟢 BAIXO


In [135]:
# Ajustar de Threshold

modelo_rf = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_rf.fit(X_train, y_train)
y_prob = modelo_rf.predict_proba(X_test)[:, 1]

print(f'{"Threshold":<12} {"F1":<8} {"Precision":<12} {"Recall":<10} {"AUC-ROC"}')
print('-' * 55)

for threshold in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]:
    y_pred_t = (y_prob >= threshold).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    classe   = str(y_test.unique().max())
    print(f'{threshold:<12} '
          f'{report[classe]["f1-score"]:<8.3f} '
          f'{report[classe]["precision"]:<12.3f} '
          f'{report[classe]["recall"]:<10.3f} '
          f'{roc_auc_score(y_test, y_prob):.3f}')

Threshold    F1       Precision    Recall     AUC-ROC
-------------------------------------------------------
0.5          0.625    0.938        0.469      0.821
0.45         0.612    0.882        0.469      0.821
0.4          0.600    0.833        0.469      0.821
0.35         0.577    0.750        0.469      0.821
0.3          0.566    0.714        0.469      0.821
0.25         0.586    0.654        0.531      0.821
0.2          0.610    0.667        0.562      0.821


In [136]:

for threshold in [0.20, 0.15, 0.10, 0.08, 0.05]:
    y_pred_t = (y_prob >= threshold).astype(int)
    report   = classification_report(y_test, y_pred_t, output_dict=True)
    classe   = str(y_test.unique().max())
    print(f'Threshold {threshold} → '
          f'Precision={report[classe]["precision"]:.3f} | '
          f'Recall={report[classe]["recall"]:.3f} | '
          f'F1={report[classe]["f1-score"]:.3f}')

Threshold 0.2 → Precision=0.667 | Recall=0.562 | F1=0.610
Threshold 0.15 → Precision=0.413 | Recall=0.594 | F1=0.487
Threshold 0.1 → Precision=0.400 | Recall=0.625 | F1=0.488
Threshold 0.08 → Precision=0.370 | Recall=0.625 | F1=0.465
Threshold 0.05 → Precision=0.278 | Recall=0.688 | F1=0.396


In [137]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score

# scorer baseado no treino
scorer_recall = make_scorer(recall_score, pos_label=y_train.unique().max())

param_grid = {
    'n_estimators'    : [100, 200, 300, 500],
    'max_depth'       : [3, 4, 5, 6],
    'learning_rate'   : [0.01, 0.05, 0.1, 0.2],
    'subsample'       : [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

resultados_grid = {}

for nome, (modelo_base, estrategia) in {
    'XGBoost (sem balanceamento)': (XGBClassifier(eval_metric='logloss', random_state=42), 'sem_smote'),
    'XGBoost (scale_pos_weight)' : (XGBClassifier(eval_metric='logloss', random_state=42), 'sem_smote'),
    'XGBoost (SMOTE)'            : (XGBClassifier(eval_metric='logloss', random_state=42), 'smote'),
}.items():

    # Prepara os dados conforme a estratégia
    if estrategia == 'smote':
        X_tr, y_tr = SMOTE(random_state=42).fit_resample(X_train, y_train)
    else:
        X_tr, y_tr = X_train, y_train

    # Grid específico por versão
    if nome == 'XGBoost (sem balanceamento)':
        grid = {**param_grid, 'scale_pos_weight': [1]}
    elif nome == 'XGBoost (scale_pos_weight)':
        grid = {**param_grid, 'scale_pos_weight': [1, 3, 5, 9]}
    else:
        grid = param_grid  # SMOTE: scale_pos_weight removido do base

    # GridSearch otimizando Recall
    gs = GridSearchCV(
        modelo_base, grid,
        scoring=scorer_recall,
        cv=5, n_jobs=-1, verbose=0
    )
    gs.fit(X_tr, y_tr)

    # Avalia com threshold 0.20
    y_prob = gs.best_estimator_.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.20).astype(int)

    report = classification_report(y_test, y_pred, output_dict=True)
    classe = str(y_train.unique().max())

    resultados_grid[nome] = {
        'melhores_params': gs.best_params_,
        'Recall'         : round(report[classe]['recall'], 3),
        'Precision'      : round(report[classe]['precision'], 3),
        'F1'             : round(report[classe]['f1-score'], 3),
        'AUC-ROC'        : round(roc_auc_score(y_test, y_prob), 3)
    }

    print(f'\n{"="*55}')
    print(f'{nome}')
    print(f'Melhores parâmetros: {gs.best_params_}')
    print(classification_report(y_test, y_pred, target_names=['NO PRAZO', 'ATRASO']))

# Comparativo final
df_grid = pd.DataFrame(resultados_grid).T
print('\nCOMPARATIVO FINAL')
print(df_grid[['Recall','Precision','F1','AUC-ROC']].sort_values('Recall', ascending=False))


XGBoost (sem balanceamento)
Melhores parâmetros: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 300, 'scale_pos_weight': 1, 'subsample': 1.0}
              precision    recall  f1-score   support

    NO PRAZO       0.95      0.98      0.97       295
      ATRASO       0.78      0.56      0.65        32

    accuracy                           0.94       327
   macro avg       0.87      0.77      0.81       327
weighted avg       0.94      0.94      0.94       327



KeyboardInterrupt: 